In [ ]:
%pip install opendatasets
import opendatasets as od
od.download_kaggle_dataset("https://www.kaggle.com/competitions/what-on-the-video/data", "")

In [ ]:
import os

from tqdm import tqdm

import zipfile
from pathlib import Path

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from ultralytics import YOLO
from sklearn.metrics import accuracy_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

# Подготовка данных

In [ ]:
DATA_DIR = '/content/what-on-the-video'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')
SAMPLE_SUBMISSION = os.path.join(DATA_DIR, 'sample_submission.csv')

CLASSES = ['animal', 'car', 'cloud', 'dance', 'fire', 'flower', 'food', 'sunset', 'water']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}
IDX_TO_CLASS = {idx: cls for cls, idx in CLASS_TO_IDX.items()}

print(f"Classes: {CLASSES}")
print(f"Train videos: {len(os.listdir(TRAIN_DIR)) if os.path.exists(TRAIN_DIR) else 0}")
print(f"Test videos: {len(os.listdir(TEST_DIR)) if os.path.exists(TEST_DIR) else 0}")

In [ ]:
def extract_frames(video_path, num_frames=3):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return []

    frame_indices = [int(total_frames * p) for p in [0.25, 0.5, 0.75]]
    frame_indices = [max(0, min(total_frames - 1, idx)) for idx in frame_indices]

    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

    cap.release()
    return frames

def load_labels():
    labels_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
    video_labels = {}
    for _, row in labels_df.iterrows():
        labels = [l.strip() for l in re.split(r"[,.]\s*", row['label'])]
        video_labels[row['file_name']] = labels
    return video_labels

In [ ]:
class VideoFrameDataset(Dataset):
    def __init__(self, video_dir, video_labels, transform=None):
        self.video_dir = video_dir
        self.video_labels = video_labels
        self.video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
        self.transform = transform
        self.samples = []

        for video_file in self.video_files:
            if video_file in video_labels:
                labels = video_labels[video_file]
                label_indices = [CLASS_TO_IDX[l] for l in labels if l in CLASS_TO_IDX]
                if label_indices:
                    self.samples.append((video_file, label_indices))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_file, label_indices = self.samples[idx]
        video_path = os.path.join(self.video_dir, video_file)
        frames = extract_frames(video_path, num_frames=3)

        if not frames:
            frames = [np.zeros((224, 224, 3), dtype=np.uint8) for _ in range(3)]

        if self.transform:
            frames = [self.transform(frame) for frame in frames]

        multi_hot = torch.zeros(len(CLASSES))
        for li in label_indices:
            multi_hot[li] = 1.0

        return torch.stack(frames), multi_hot

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
video_labels = load_labels()
full_dataset = VideoFrameDataset(TRAIN_DIR, video_labels, transform=transform)

In [ ]:
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

val_dataset.dataset.transform = val_transform

In [ ]:
BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# Модель

In [ ]:
class FrameClassifier(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
        self.backbone = YOLO('yolov8n-cls.pt').model
        self.backbone = self.backbone.model

        in_features = self.backbone[-1].cv2[2].in_features
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )
        self.backbone[-1].cv2[2] = nn.Identity()
        self.backbone[-1].cv3[2] = nn.Identity()

    def forward(self, x):
        batch_size, num_frames, c, h, w = x.shape
        x = x.view(batch_size * num_frames, c, h, w)

        features = self.backbone(x)
        logits = self.classifier(features)

        logits = logits.view(batch_size, num_frames, -1)
        logits = logits.mean(dim=1)

        return logits

In [ ]:
model = FrameClassifier(num_classes=len(CLASSES)).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

In [ ]:
def train_epoch(loader):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc='Training')
    for frames, labels in pbar:
        frames, labels = frames.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * frames.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

        pbar.set_postfix({'Loss': f'{total_loss / len(loader.dataset):.4f}'})

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    class_acc = []
    for i in range(len(CLASSES)):
        mask = all_labels[:, i] != -1
        if mask.sum() > 0:
            acc = accuracy_score(all_labels[mask, i], all_preds[mask, i])
            class_acc.append(acc)

    return total_loss / len(loader.dataset), np.mean(class_acc)

def validate(loader):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for frames, labels in pbar:
            frames, labels = frames.to(DEVICE), labels.to(DEVICE)
            outputs = model(frames)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * frames.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

            pbar.set_postfix({'Loss': f'{total_loss / len(loader.dataset):.4f}'})

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    class_acc = []
    for i in range(len(CLASSES)):
        mask = all_labels[:, i] != -1
        if mask.sum() > 0:
            acc = accuracy_score(all_labels[mask, i], all_preds[mask, i])
            class_acc.append(acc)

    return total_loss / len(loader.dataset), np.mean(class_acc)

In [ ]:
EPOCHS = 20
best_val_acc = 0.0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 50)

    train_loss, train_acc = train_epoch(train_loader)
    val_loss, val_acc = validate(val_loader)

    scheduler.step()

    print(f"Train Loss: {train_loss:.4f}, Train Class-wise Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Class-wise Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Saved best model with val class-wise accuracy: {val_acc:.4f}")

print(f"\nBest validation class-wise accuracy: {best_val_acc:.4f}")

# Предсказания

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

test_videos = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.mp4')])
print(f"Test videos: {len(test_videos)}")

predictions = []

for video_file in tqdm(test_videos):
    video_path = os.path.join(TEST_DIR, video_file)
    frames = extract_frames(video_path, num_frames=3)

    if not frames:
        pred_labels = []
    else:
        processed_frames = []
        for frame in frames:
            frame_tensor = val_transform(frame).unsqueeze(0)
            processed_frames.append(frame_tensor)

        frames_tensor = torch.stack(processed_frames, dim=0).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            outputs = model(frames_tensor)
            probs = torch.sigmoid(outputs).squeeze().cpu().numpy()

        if len(probs.shape) == 0:
            probs = np.array([probs])

        if probs.shape[0] == len(CLASSES):
            pred_labels = [CLASSES[i] for i, p in enumerate(probs) if p > 0.5]
            if not pred_labels:
                pred_labels = [CLASSES[np.argmax(probs)]]
        else:
            pred_labels = [CLASSES[0]]

    predictions.append({
        'index': len(predictions),
        'file_name': video_file,
        'label': ', '.join(pred_labels)
    })

submission_df = pd.DataFrame(predictions)
submission_df.to_csv('submission.csv', index=False)
print("\nSubmission saved to submission.csv")
print(submission_df.head())

In [ ]:
!head submission.csv